# colab_12 — Geneformer CPT evals #1 & #2 (aggregated regime, v2 · seed 0)

Does the aggregated CPT run (colab_11 v2) actually **help the biology**, not just move the embedding? This notebook runs the two pre-registered evals that read the glia embedding directly:

- **Eval #1** — substate linear probe (does CPT sharpen within-lineage state structure?)
- **Eval #2** — APOE-carrier recovery within astrocytes and within microglia (the Stanton-core axis)

Both are scored strictly by `docs/EVALUATION_CONTRACT.md`: improvement over the zero-shot baseline, on the **same global held-out donors**, against bands fixed before any CPT result existed.

**Two extraction points, by design.** The head-ablation diagnostic showed v2's loss gain is concentrated in the head + last encoder layer — structurally *downstream* of the standard cell-embedding readout (`emb_layer=-1`, second-to-last layer). So every eval here is run at **both** `emb_layer=-1` (the readout the whole pipeline uses) **and** `emb_layer=0` (the last layer). The −1-vs-0 gap is the eval-space read on layer 11's absorbed share — a partial view: it captures layer 11 but *not* the head, whose contribution lives only in reconstruction space and is unreachable as an embedding.

Eval #3 / detector #2 (catastrophic forgetting) are **not** here — they need a non-AD reference dataset and are a separate deliverable (colab_13).

## 1 — Setup

### 1a — Mount Drive, clone/pull the repo, install the Geneformer environment, set run flags

Same lean Geneformer-only stack as colab_09/11 (rides Colab's native numpy-2 base; scGPT is not installed). A GPU is required — this notebook re-embeds the substrate, which is not viable on CPU.

`SMOKE` is the one live switch: committed `False`. When `True`, the notebook subsamples the substrate to a few thousand cells *after* the split is assigned, so the whole path (tokenize → embed at layer 0 → probe → k-NN) can be exercised cheaply before paying for the full embedding. Every derived path is suffixed `_SMOKE` so a smoke run can waste time but never overwrite a real output.

In [1]:
import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/ad-glia-fm-prep"
os.makedirs(DRIVE_ROOT, exist_ok=True)

REPO_URL  = "https://github.com/pavlemic/ad-glia-fm-prep.git"
REPO_PATH = "/content/ad-glia-fm-prep"
if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(["git", "-C", REPO_PATH, "pull"], check=True)
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

assert sys.version_info >= (3, 10), f"Geneformer needs Python >=3.10, got {sys.version_info[:2]}."

!pip install -r {REPO_PATH}/requirements_geneformer.txt

GENEFORMER_REPO = "/content/Geneformer"
if not os.path.exists(GENEFORMER_REPO):
    !git lfs install
    !git clone https://huggingface.co/ctheodoris/Geneformer {GENEFORMER_REPO}
!cd {GENEFORMER_REPO} && pip install .
# torchao (Colab-preinstalled, unused) hard-raises inside peft dispatch below its floor -- remove it.
!pip uninstall -y torchao -q

GENEFORMER_COMMIT = subprocess.run(
    ["git", "-C", GENEFORMER_REPO, "rev-parse", "HEAD"],
    capture_output=True, text=True).stdout.strip()

import torch
assert torch.cuda.is_available(), "no CUDA GPU -- select a GPU runtime before re-embedding"
print("Python:", sys.version.split()[0], "| Geneformer commit:", GENEFORMER_COMMIT,
      "| GPU:", torch.cuda.get_device_name(0))

# --- the one live switch; ALL variable output paths derive from SUFFIX ---
SMOKE   = False
SMOKE_N_PER_GROUP = 40          # cells kept per (lineage x split x substate) group when SMOKE
SUFFIX  = "_SMOKE" if SMOKE else ""
SEED    = 0
from datetime import date
TODAY   = date.today().isoformat()
print(f"SMOKE={SMOKE} | output suffix='{SUFFIX}'")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Processing /content/Geneformer
  Preparing metadata (setup.py) ... done
  Created wheel for geneformer: filename=geneformer-0.1.0-py3-none-any.whl size=2980779 sha256=712c83480a92ce1e18e77127ce6bcb524f8600390c1d9c492d91238db0534300
  Stored in directory: /tmp/pip-ephem-wheel-cache-id8znheb/wheels/2d/46/09/b7648deddd8f78de5e5a2785bbf00a6b4d1246c6d434192a76
Successfully built geneformer
  Attempting uninstall: geneformer
    Found existing installation: geneformer 0.1.0
    Uninstalling geneformer-0.1.0:
      Successfully uninstalled geneformer-0.1.0
Python: 3.12.13 | Geneformer commit: 04c2b2e84da7c0f385c3f9ad8f3ec24bab6650e5 | GPU: NVIDIA A100-SXM4-80GB
SMOKE=False | output suffix=''


> **Interpretation — environment installed, real run confirmed (1a).** The environment resolved cleanly against the same pinned stack as the prior dry run (transformers 4.57.0, peft 0.19.1, datasets 5.0.0 all already satisfied in this runtime), the Geneformer repo installed at the same commit (`04c2b2e8`) used throughout this project, and `torchao` was removed without incident (it hard-raises inside `peft`'s dispatch below its floor if left in). The GPU resolved to an A100-SXM4-80GB. The final printed line, `SMOKE=False | output suffix=''`, is the confirmation this is the real, full-scale run: every output path this notebook writes from here on is the permanent, non-suffixed location, not the `_SMOKE`-suffixed scratch path a dry run would have used.

### 1b — pip freeze + env JSON (records the exact eval-run stack)

Same freeze-then-document discipline as colab_09/11 §2a: snapshot the resolved versions + Geneformer commit + GPU/CUDA for the Methods record. This matters here specifically because §4b re-merges the v2 LoRA adapter, and `peft`'s merge semantics are already known to be correctness-critical and silently driftable if the resolved version skews (see `docs/ASSUMPTIONS.md`).

In [2]:
import json, platform, subprocess, sys

NOTEBOOK_ID = "colab_12"
VERSIONS_DIR = os.path.join(REPO_PATH, "outputs", "software_versions")
os.makedirs(VERSIONS_DIR, exist_ok=True)

FREEZE_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_pip_freeze.txt")
!pip freeze > {FREEZE_PATH}

def _run(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True, check=True).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None

def _ver(mod):
    try:
        return __import__(mod).__version__
    except Exception:
        return None

env_snapshot = {
    "notebook_id":    NOTEBOOK_ID,
    "date":           TODAY,
    "python_version": sys.version,
    "platform":       platform.platform(),
    "os_release":     platform.release(),
    "gpu":            _run(["nvidia-smi", "-L"]),
    "cuda":           _run(["nvcc", "--version"]),
    "git_commit":     _run(["git", "-C", REPO_PATH, "rev-parse", "HEAD"]),
    "geneformer_commit":    GENEFORMER_COMMIT,
    "scanpy_version":       _ver("scanpy"),
    "anndata_version":      _ver("anndata"),
    "torch_version":        _ver("torch"),
    "transformers_version": _ver("transformers"),
    "peft_version":         _ver("peft"),
    "accelerate_version":   _ver("accelerate"),
    "datasets_version":     _ver("datasets"),
    "geneformer_version":   _ver("geneformer"),
}
ENV_JSON_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_env.json")
with open(ENV_JSON_PATH, "w") as f:
    json.dump(env_snapshot, f, indent=2)
print(json.dumps(env_snapshot, indent=2))

/tmp/ipykernel_8354/1404772190.py:18: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  return __import__(mod).__version__
/tmp/ipykernel_8354/1404772190.py:18: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  return __import__(mod).__version__


{
  "notebook_id": "colab_12",
  "date": "2026-07-20",
  "python_version": "3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "os_release": "6.6.122+",
  "gpu": "GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-002c0715-1948-eaea-69d8-001976cbab1a)",
  "cuda": "nvcc: NVIDIA (R) Cuda compiler driver\nCopyright (c) 2005-2025 NVIDIA Corporation\nBuilt on Fri_Feb_21_20:23:50_PST_2025\nCuda compilation tools, release 12.8, V12.8.93\nBuild cuda_12.8.r12.8/compiler.35583870_0",
  "git_commit": "0ebb5118bebb95153435085c1851eef27735601b",
  "geneformer_commit": "04c2b2e84da7c0f385c3f9ad8f3ec24bab6650e5",
  "scanpy_version": "1.12.2",
  "anndata_version": "0.13.2",
  "torch_version": "2.11.0+cu128",
  "transformers_version": "4.57.0",
  "peft_version": "0.19.1",
  "accelerate_version": "1.14.0",
  "datasets_version": "5.0.0",
  "geneformer_version": null
}


> **Interpretation — env snapshot recorded (1b).** The printed JSON is the recorded software/hardware snapshot for this exact run: Python 3.12.13, an A100-80GB GPU, CUDA 12.8, git commit `0ebb511` (the exact repo state this notebook was pulled from), Geneformer commit `04c2b2e8`, and the exact pinned versions of scanpy/anndata/torch/transformers/peft/accelerate/datasets. `geneformer_version` prints `null` because the installed `geneformer` package doesn't expose a `__version__` attribute — the helper's try/except silently returns `None` for it — but the commit hash captured separately is the real record of which Geneformer snapshot ran. Both this JSON and the parallel pip-freeze text file are the artifacts that make this run's exact software stack reproducible later.

## 2 — Load the substrate and validate the schema

### 2a — Rebuild the glia substrate (deterministic; same `cell_index` as the saved embeddings)

The saved embeddings carry a `cell_index` that was assigned when colab_11 concatenated the microglia and astrocyte subsets. Rebuilding `glia` here from the *same* two subset files, in the *same* order, reproduces that `cell_index` exactly — it is the key every embedding matrix is realigned to. We need the raw-counts object (not just the embeddings) because re-tokenizing for the layer-0 pass requires the counts.

`region` is kept here directly: colab_07/08 built each subset as a `.copy()` of the colab_05 parent without dropping obs columns, so `region` (needed for eval #2's confound audit) already rides along on the subset files — no separate join required.

In [3]:
import gc
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import scipy.sparse as sp
sc.settings.verbosity = 1

MICRO_PATH = os.path.join(DRIVE_ROOT, "micro_subset", "micro_subset.h5ad")
ASTRO_PATH = os.path.join(DRIVE_ROOT, "astro_subset", "astro_subset.h5ad")
for p in (MICRO_PATH, ASTRO_PATH):
    if not os.path.exists(p):
        raise FileNotFoundError(f"missing labelled subset {p} (colab_07 / colab_08 output)")

micro = sc.read_h5ad(MICRO_PATH)
astro = sc.read_h5ad(ASTRO_PATH)
assert list(micro.var_names) == list(astro.var_names), "gene panels differ between subsets"
micro.obs["lineage"] = "microglia"
astro.obs["lineage"] = "astrocyte"
KEEP_OBS = ["lineage", "substate", "apoe_carrier", "study_id", "donor_id", "region", "total_counts"]
micro.obs = micro.obs[[c for c in KEEP_OBS if c in micro.obs.columns]].copy()
astro.obs = astro.obs[[c for c in KEEP_OBS if c in astro.obs.columns]].copy()
glia = ad.concat([micro, astro], join="inner", index_unique="-")
del micro, astro; gc.collect()
glia.obs["cell_index"] = np.arange(glia.n_obs)
print("combined glia:", glia.shape)
print("lineage:", glia.obs["lineage"].value_counts().to_dict())
print("substate:", glia.obs["substate"].value_counts(dropna=False).to_dict())
print("apoe_carrier:", glia.obs["apoe_carrier"].value_counts(dropna=False).to_dict())

# raw-counts guard -- Geneformer rank-encodes RAW counts.
_idx = np.random.default_rng(0).choice(glia.n_obs, size=min(2000, glia.n_obs), replace=False)
Xs = glia.X[_idx]
data = Xs.data if sp.issparse(Xs) else np.asarray(Xs).ravel()
frac_int = float(np.mean(np.mod(data, 1) == 0)) if data.size else 1.0
assert frac_int >= 0.99, f".X is not raw counts (int frac {frac_int:.3f})"
assert glia.n_obs == 142588, f"expected 142,588-cell substrate, got {glia.n_obs} -- subset files changed"

combined glia: (142588, 26514)
lineage: {'astrocyte': 87783, 'microglia': 54805}
substate: {'resting': 48147, 'reactive': 28465, 'intermediate': 28022, 'homeostatic': 25845, 'activated': 12109}
apoe_carrier: {'noncarrier': 70169, 'carrier': 49831, 'e2': 22588}


> **Interpretation — substrate rebuilt (2a).** The rebuilt glia substrate is 142,588 cells × 26,514 genes, split 87,783 astrocyte / 54,805 microglia — an exact match to the frozen substrate every foundation-model notebook in this project uses. The printed substate dict combines both lineages' independent labeling schemes into one obs column: astrocyte contributes resting (48,147) and reactive (28,465), microglia contributes homeostatic (25,845) and activated (12,109), and both lineages' "intermediate" bucket is summed together (16,851 micro + 11,171 astro = 28,022) since it isn't used in the binary probes downstream. All five numbers sum to 142,588 — the full substrate is accounted for, no cell dropped silently. `apoe_carrier` splits noncarrier 70,169 / carrier 49,831 / e2 22,588 (also summing to 142,588); `e2` is deliberately its own bucket rather than folded into "noncarrier," because carriers of the E2 allele without E4 are excluded from the primary binary APOE axis by design. The two asserts that ran silently — the raw-counts guard on 2,000 sampled cells, and the exact 142,588-cell count — both passed, meaning the input tensors are still integer raw counts (required for Geneformer's rank encoding) and the labelled subset files haven't drifted from the substrate every other notebook is anchored to.

### 2b — Fail loud on the obs schema eval #1/#2 depend on

Eval #2's contract *mandates* reporting the study **and region** composition of the cells driving any APOE recovery, and eval #1 needs `substate`. This cell asserts every such column is present and non-null and that the categorical values are the expected ones — so a missing or malformed label stops the run here instead of silently degrading a downstream audit. `region` in particular must have survived the subset inheritance (§2a); if it is absent or partly null, the assert points back to the colab_07/08 outputs.

In [4]:
REQUIRED_OBS = ["cell_index", "lineage", "substate", "apoe_carrier", "study_id", "region", "donor_id"]
missing_cols = [c for c in REQUIRED_OBS if c not in glia.obs.columns]
assert not missing_cols, (
    f"substrate missing required obs columns: {missing_cols} "
    "(region is inherited from the colab_07/08 subsets -- check they carry it)")

for col in REQUIRED_OBS:
    n_null = int(pd.isna(glia.obs[col]).sum())
    assert n_null == 0, f"{col} has {n_null} null values -- eval audits require it complete"

assert set(glia.obs["lineage"]) == {"microglia", "astrocyte"}, "unexpected lineage values"
assert set(glia.obs["substate"]) <= {"homeostatic", "activated", "resting", "reactive", "intermediate"}, \
    f"unexpected substate values: {set(glia.obs['substate'])}"
assert set(glia.obs["apoe_carrier"]) <= {"carrier", "noncarrier", "e2"}, \
    f"unexpected apoe_carrier values: {set(glia.obs['apoe_carrier'])}"
print("obs schema OK | region x study:")
print(pd.crosstab(glia.obs["region"], glia.obs["study_id"]))

obs schema OK | region x study:
study_id         Li2025  SEA-AD  Haney2024
region                                    
MTG                   0   75634          0
temporal cortex   46870       0          0
unknown               0       0      20084


> **Interpretation — schema check and the Haney "unknown" region caveat (2b).** The schema assert passed silently, so every one of the 7 required obs columns (`cell_index`, `lineage`, `substate`, `apoe_carrier`, `study_id`, `region`, `donor_id`) is present with zero nulls across all 142,588 cells, and every categorical value falls inside the expected vocabulary. The printed region × study crosstab shows each study contributes cells from exactly one region label: Li2025 → "temporal cortex" (46,870 cells), SEA-AD → "MTG" (75,634 cells), and Haney2024 → the literal string "unknown" (20,084 cells). This is expected: Haney2024's actual anatomical region was never present in its deposited metadata, so every Haney cell carries the placeholder value "unknown" rather than a true region label. Because "unknown" is a non-null string, this schema assert legitimately passes — but it means eval #2's later study × region confound audit (7a) cannot distinguish regional structure within Haney; all 20,084 of its cells fall into one uninformative "unknown" bucket. Read the 7a table with that in mind.

## 3 — Held-out split verification

### 3a — Rebuild the frozen donor split and hard-stop on any mismatch

The evals must score on the *same* held-out donors every regime and FM uses. The split is a deterministic function of the unchanged colab_05 parent (study-stratified `train_test_split`, seed searched over `range(200)` maximizing the worst-case test-donor margin), so rebuilding it here must reproduce the reference exactly. Any drift in seed, margin, or split counts is a hard stop — never a silently re-drawn split. Reference (colab_11, 2026-07-10): seed 32, worst-case margin 10, donors 101/22/22, cells 94963/23824/23801, test study fractions SEA-AD 0.567 / Li2025 0.30 / Haney2024 0.133.

If `SMOKE`, the subsample is drawn **here, after** the split is assigned, stratified by (lineage × split × substate) so every group the evals need stays populated.

In [5]:
import json
from sklearn.model_selection import train_test_split

DONOR_META_PATH = os.path.join(REPO_PATH, "outputs", "donor_metadata.csv")
SPLIT_FRACS     = {"train": 0.70, "val": 0.15, "test": 0.15}
SEED_POOL       = range(200)

assert os.path.exists(DONOR_META_PATH), f"need committed donor metadata {DONOR_META_PATH} (colab_11 minted it)"
donor_meta = pd.read_csv(DONOR_META_PATH, dtype=str)
sub_donors = set(glia.obs["donor_id"].astype(str))
donor_meta = donor_meta[donor_meta["donor_id"].isin(sub_donors)].reset_index(drop=True)
assert donor_meta["donor_id"].is_unique, "a donor_id maps to >1 metadata row"

def _study_split(seed):
    d_tr, d_te = train_test_split(donor_meta["donor_id"], test_size=SPLIT_FRACS["test"],
                                  random_state=seed, stratify=donor_meta["study_id"])
    strat_tr = donor_meta.set_index("donor_id").loc[d_tr.values, "study_id"]
    d_tr, d_va = train_test_split(d_tr, test_size=SPLIT_FRACS["val"] / (SPLIT_FRACS["train"] + SPLIT_FRACS["val"]),
                                  random_state=seed, stratify=strat_tr)
    return set(d_tr), set(d_va), set(d_te)

def _test_margin(d_test):
    m = donor_meta[donor_meta["donor_id"].isin(d_test)]
    return int(min((m["apoe_carrier"] == "carrier").sum(), (m["apoe_carrier"] == "noncarrier").sum(),
                   (m["diagnosis"] == "AD").sum(), (m["diagnosis"] == "Control").sum()))

best_seed = max(SEED_POOL, key=lambda s: _test_margin(_study_split(s)[2]))
d_train, d_val, d_test = _study_split(best_seed)
margin = _test_margin(d_test)
split_map = {**{d: "train" for d in d_train}, **{d: "val" for d in d_val}, **{d: "test" for d in d_test}}
glia.obs["split"] = glia.obs["donor_id"].astype(str).map(split_map).astype("category")
assert not glia.obs["split"].isna().any(), "some cells' donor received no split assignment"

n_donors = {k: int(sum(1 for v in split_map.values() if v == k)) for k in ("train", "val", "test")}
n_cells  = glia.obs["split"].value_counts().to_dict()
test_by_study = glia.obs.loc[glia.obs["split"] == "test", "study_id"].value_counts(normalize=True).round(3).to_dict()
print(f"seed {best_seed} | margin {margin} | donors {n_donors} | cells {n_cells}")
print("test study fractions:", test_by_study)

# HARD STOP: match the frozen reference exactly (standing split-verification rule)
assert best_seed == 32,  f"seed {best_seed} != reference 32 -- split drifted, do NOT proceed"
assert margin == 10,     f"margin {margin} != reference 10 -- split drifted"
assert n_donors == {"train": 101, "val": 22, "test": 22}, f"donor counts {n_donors} != reference"
assert n_cells.get("train") == 94963 and n_cells.get("val") == 23824 and n_cells.get("test") == 23801, \
    f"cell counts {n_cells} != reference 94963/23824/23801"
print("split verification PASSED -- matches the frozen reference")

# --- SMOKE subsample: AFTER the split is assigned; keep every (lineage x split x substate) group ---
if SMOKE:
    keep = (glia.obs.groupby(["lineage", "split", "substate"], observed=True)
            .apply(lambda g: g.sample(min(len(g), SMOKE_N_PER_GROUP), random_state=SEED))
            .index.get_level_values(-1))
    glia = glia[glia.obs.index.isin(keep)].copy()
    print(f"[SMOKE] subsampled to {glia.n_obs} cells | split:", glia.obs["split"].value_counts().to_dict())

seed 32 | margin 10 | donors {'train': 101, 'val': 22, 'test': 22} | cells {'train': 94963, 'val': 23824, 'test': 23801}
test study fractions: {'SEA-AD': 0.567, 'Li2025': 0.3, 'Haney2024': 0.133}
split verification PASSED -- matches the frozen reference


> **Interpretation — split verification passed (3a).** The donor split is not drawn fresh here — it is a deterministic function of the same donor metadata and search procedure the CPT training notebook used (search seeds 0–199, pick whichever study-stratified 70/15/15 split maximizes the worst-case count across carrier/noncarrier/AD/control in the test donors), so re-running it against the unchanged substrate must reproduce the exact same split. It did: seed 32, worst-case margin 10 (donors), 101/22/22 donors and 94,963/23,824/23,801 cells train/val/test, with the test set's per-study composition (SEA-AD 56.7%, Li2025 30.0%, Haney2024 13.3%) matching the frozen reference to three decimals. The five hard asserts immediately after the print (seed==32, margin==10, exact donor counts, exact cell counts) all passed silently — if any one of them had failed, the notebook would have raised and stopped rather than silently proceeding on a drifted split. This is the load-bearing guarantee that every eval in this notebook scores on literally the same held-out donors the CPT training run never saw, and that any other regime/FM notebook comparing against these numbers is comparing apples to apples.

## 4 — Produce the last-layer (`emb_layer=0`) embeddings

### 4a — Tokenize the substrate (deterministic; identical encoding to the CPT run)

The layer-0 embeddings do not exist yet — colab_11 saved only the layer −1 embeddings and the LoRA adapter, and the tokenized dataset was ephemeral. So the substrate is re-tokenized here with the exact same encoding colab_09/11 used (V2, 4096 input size, GC104M dictionaries), gated on APOE surviving the vocabulary. Re-tokenizing is deterministic, so it reproduces the CPT run's inputs cell-for-cell.

In [6]:
from geneformer import ENSEMBL_DICTIONARY_FILE, TOKEN_DICTIONARY_FILE, TranscriptomeTokenizer
import pickle
import geneformer.tokenizer as _gftok

with open(ENSEMBL_DICTIONARY_FILE, "rb") as f:
    symbol_to_ensembl = pickle.load(f)
with open(TOKEN_DICTIONARY_FILE, "rb") as f:
    token_dictionary = pickle.load(f)

glia.var["ensembl_id"] = [symbol_to_ensembl.get(s) for s in glia.var_names]
in_vocab = glia.var["ensembl_id"].map(lambda e: (e in token_dictionary) if e is not None else False)
apoe_e = symbol_to_ensembl.get("APOE")
assert apoe_e is not None and apoe_e in token_dictionary, (
    "APOE not tokenizable -- eval #2 impossible for Geneformer (pre-registered hard fail)")
print(f"genes in Geneformer vocab: {int(in_vocab.sum())} / {glia.n_vars}")

glia.obs["n_counts"] = np.asarray(glia.X.sum(axis=1)).ravel()
assert (glia.obs["n_counts"] > 0).all(), "cells with zero counts present"

TOK_IN_DIR  = f"/content/gf_token_in{SUFFIX}"
TOK_OUT_DIR = f"/content/gf_token_out{SUFFIX}"
os.makedirs(TOK_IN_DIR, exist_ok=True); os.makedirs(TOK_OUT_DIR, exist_ok=True)
glia.write_h5ad(os.path.join(TOK_IN_DIR, "glia.h5ad"))

CUSTOM_ATTRS = {c: c for c in ["cell_index", "split", "lineage", "substate", "apoe_carrier", "study_id", "donor_id"]}

# same RangeIndex monkeypatch colab_11 needed: tokenize_anndata positional-indexes var by an
# integer array against a non-integer index; reset it so upstream's assumed behaviour holds.
_orig_sum = _gftok.sum_ensembl_ids
def _sum_rangeindex_patch(*a, **k):
    r = _orig_sum(*a, **k); r.var.index = pd.RangeIndex(r.n_vars); return r
_gftok.sum_ensembl_ids = _sum_rangeindex_patch

tk = TranscriptomeTokenizer(CUSTOM_ATTRS, nproc=4)
tk.tokenize_data(TOK_IN_DIR, TOK_OUT_DIR, f"glia_evals{SUFFIX}", file_format="h5ad")
TOKENIZED_DATASET = os.path.join(TOK_OUT_DIR, f"glia_evals{SUFFIX}.dataset")
print("tokenized ->", TOKENIZED_DATASET)

genes in Geneformer vocab: 16792 / 26514
Tokenizing /content/gf_token_in/glia.h5ad
/content/gf_token_in/glia.h5ad has no column attribute 'filter_pass'; tokenizing all cells.
Creating dataset.
tokenized -> /content/gf_token_out/glia_evals.dataset


> **Interpretation — substrate tokenized (4a).** 16,792 of the substrate's 26,514 genes (63.3%) map into Geneformer's tokenizer vocabulary — identical to the earlier zero-shot baseline notebook's number, confirming the same gene panel and the same Ensembl-mapping/vocabulary dictionaries are in play here as in every prior Geneformer notebook. The APOE-tokenizable hard-assert passed silently: if APOE itself had fallen outside the vocabulary, eval #2 — the entire Stanton-core axis — would be impossible for this FM, and the notebook would have stopped here rather than produce an untrustworthy result downstream. The tokenizer's own message ("no column attribute 'filter_pass'; tokenizing all cells") is expected and benign — it only means no cells were pre-flagged for exclusion, nothing is being silently dropped. This cell's output dataset is written to a local, ephemeral scratch path — deliberately not saved anywhere persistent, since only the resulting embeddings (not the intermediate tokenized dataset) are treated as artifacts worth keeping.

### 4b — Extract layer-0 embeddings: frozen base (zero-shot L0) and merged v2 adapter (CPT L0)

Two passes of Geneformer's `EmbExtractor` at `emb_layer=0` (the last encoder layer). The zero-shot L0 runs the frozen base model; the CPT L0 reloads that same base, attaches the saved v2 LoRA adapter, and merges it before extracting — reproducing colab_11's post-CPT model, read one layer deeper than detector #1 did. Each output is written `_L0`-suffixed and guarded: if the file already exists (a re-run), it is loaded rather than recomputed.

In [7]:
from peft import PeftModel
from transformers import BertForMaskedLM
from geneformer import EmbExtractor

GF_DIR    = os.path.join(DRIVE_ROOT, "geneformer")
MODEL_DIR = os.path.join(GENEFORMER_REPO, "Geneformer-V2-104M")
ADAPTER_DIR = os.path.join(GF_DIR, "cpt_aggregated_v2_seed0_adapter")   # colab_11 v2 output
assert os.path.exists(MODEL_DIR),   f"base model missing: {MODEL_DIR}"
assert os.path.exists(ADAPTER_DIR), f"v2 LoRA adapter missing on Drive: {ADAPTER_DIR}"

EMB_LABELS = ["cell_index", "split", "lineage", "substate", "apoe_carrier", "study_id", "donor_id"]
ZS_L0_PATH  = os.path.join(GF_DIR, f"glia_geneformer_zeroshot_L0{SUFFIX}.h5ad")
CPT_L0_PATH = os.path.join(GF_DIR, f"glia_geneformer_cpt_aggregated_v2_seed0_L0{SUFFIX}.h5ad")

def _extract_L0(model_dir, tag, out_h5ad):
    """Run EmbExtractor at emb_layer=0 on a checkpoint dir, save an aligned embedding h5ad."""
    ee = EmbExtractor(model_type="Pretrained", num_classes=0, emb_mode="cell",
                      max_ncells=None, emb_layer=0, emb_label=EMB_LABELS,
                      forward_batch_size=64, nproc=4)
    work = f"/content/gf_emb_{tag}{SUFFIX}"; os.makedirs(work, exist_ok=True)
    df = ee.extract_embs(model_dir, TOKENIZED_DATASET, work, f"glia_{tag}")
    emb_cols = [c for c in df.columns if c not in EMB_LABELS]
    df = df.set_index("cell_index").reindex(glia.obs["cell_index"].values)
    assert df[emb_cols].notna().all().all(), f"{tag}: rows missing after cell_index realignment"
    a = ad.AnnData(X=df[emb_cols].to_numpy(dtype=np.float32),
                   obs=glia.obs[EMB_LABELS].reset_index(drop=True))
    a.write_h5ad(out_h5ad)
    print(f"{tag} L0 -> {out_h5ad}  {a.shape}")

# zero-shot L0 (frozen base)
if os.path.exists(ZS_L0_PATH):
    print("zero-shot L0 exists, skipping:", ZS_L0_PATH)
else:
    _extract_L0(MODEL_DIR, "zeroshot_L0", ZS_L0_PATH)

# CPT L0 (base + v2 adapter, merged)
if os.path.exists(CPT_L0_PATH):
    print("CPT L0 exists, skipping:", CPT_L0_PATH)
else:
    base = BertForMaskedLM.from_pretrained(MODEL_DIR)
    merged = PeftModel.from_pretrained(base, ADAPTER_DIR).merge_and_unload()
    MERGED_DIR = f"/content/gf_v2_merged{SUFFIX}"; os.makedirs(MERGED_DIR, exist_ok=True)
    merged.save_pretrained(MERGED_DIR)
    base.config.to_json_file(os.path.join(MERGED_DIR, "config.json"))
    _extract_L0(MERGED_DIR, "cpt_L0", CPT_L0_PATH)

/usr/local/lib/python3.12/dist-packages/pyarrow/compute.py:230: FutureWarning: Specifying null_placement in SortOptions is deprecated as of 25.0.0. Specify null_placement per sort_key instead.
  return options_class(*args, **kwargs)


  0%|          | 0/2228 [00:00<?, ?it/s]

/usr/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


zeroshot_L0 L0 -> /content/drive/MyDrive/ad-glia-fm-prep/geneformer/glia_geneformer_zeroshot_L0.h5ad  (142588, 768)


  0%|          | 0/2228 [00:00<?, ?it/s]

/usr/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


cpt_L0 L0 -> /content/drive/MyDrive/ad-glia-fm-prep/geneformer/glia_geneformer_cpt_aggregated_v2_seed0_L0.h5ad  (142588, 768)


> **Interpretation — layer-0 embeddings extracted (4b).** Two full forward passes of the tokenized 142,588-cell dataset through Geneformer’s embedding extractor at `emb_layer=0` (the last encoder layer, one layer deeper than the pipeline’s usual `emb_layer=-1` readout) — once through the frozen base model (zero-shot L0) and once through the base model with the aggregated v2 LoRA adapter merged in (CPT L0). Both completed and both output matrices came out `(142588, 768)`: 142,588 rows (one per cell, matching the substrate exactly) by 768 columns (Geneformer’s embedding dimensionality). That match is not just cosmetic — inside `_extract_L0`, the embeddings are re-indexed by `cell_index` and asserted to have zero missing rows after alignment, so the fact both prints appeared confirms every one of the 142,588 cells got a real embedding vector from both models. The CLS/EOS warnings are Geneformer’s own bookkeeping (those special tokens are correctly excluded when it averages a cell’s gene-token embeddings into one vector) and "Transforming to str index" is a routine anndata type-coercion notice — neither indicates a problem. Each pass took roughly 1 hour 43 minutes on the A100 (2,228 batches of 64 cells), as observed live during the run — that timing is not fully recoverable from this cell’s saved output alone, since the tqdm progress-bar widget only serialized its initial 0% state when the notebook was saved, not the completed elapsed time. It is still the first real wall-clock anchor recorded for this exact operation (full-substrate embedding extraction, no backprop) in this project, just not one a reader can re-derive from this file by itself.

## 5 — Assemble the four embedding matrices

### 5a — Load L−1 (existing) and L0 (new), align all four to `cell_index`

Four matrices over the same cells: zero-shot and CPT, each at `emb_layer=-1` (loaded from the existing colab_09 / colab_11 outputs) and at `emb_layer=0` (produced above). All are realigned to the substrate's `cell_index` so a row is the same cell everywhere. Under `SMOKE`, the L−1 files are subset to the smoke cells; the L0 files already are.

In [8]:
def _load_aligned(path):
    a = sc.read_h5ad(path)
    df = pd.DataFrame(a.X, index=a.obs["cell_index"].values)
    df = df.reindex(glia.obs["cell_index"].values)
    assert df.notna().all().all(), f"{path}: rows missing after cell_index alignment"
    return df.to_numpy(dtype=np.float32)

ZS_L1_PATH  = os.path.join(GF_DIR, "glia_geneformer_zeroshot.h5ad")                       # colab_09
CPT_L1_PATH = os.path.join(GF_DIR, "glia_geneformer_cpt_aggregated_v2_seed0.h5ad")         # colab_11 v2
for p in (ZS_L1_PATH, CPT_L1_PATH):
    assert os.path.exists(p), f"missing existing L-1 embedding {p}"

EMB = {
    ("zeroshot", "L-1"): _load_aligned(ZS_L1_PATH),
    ("cpt",      "L-1"): _load_aligned(CPT_L1_PATH),
    ("zeroshot", "L0"):  _load_aligned(ZS_L0_PATH),
    ("cpt",      "L0"):  _load_aligned(CPT_L0_PATH),
}
for k, v in EMB.items():
    print(k, v.shape)
assert len({v.shape for v in EMB.values()}) == 1, "embedding matrices differ in shape"

# shared masks / labels for the evals
obs        = glia.obs.reset_index(drop=True)
is_train   = (obs["split"] == "train").to_numpy()
is_test    = (obs["split"] == "test").to_numpy()
lineage    = obs["lineage"].to_numpy()
substate   = obs["substate"].to_numpy()
apoe       = obs["apoe_carrier"].to_numpy()
LAYERS     = ["L-1", "L0"]

def band_probe(delta_pp):    # eval #1 substate-probe threshold -- eval #2 k-NN uses band_knn below (different regression cutoff)
    if delta_pp < -2: return "regression"
    if delta_pp >= 10: return "decisive"
    if delta_pp >= 5:  return "meaningful"
    return "noise"

def band_knn(delta_pp):
    if delta_pp < -3: return "regression"
    if delta_pp >= 10: return "decisive"
    if delta_pp >= 5:  return "meaningful"
    return "noise"

def band_sil(delta):
    if delta < 0: return "regression"
    if delta >= 0.10: return "decisive"
    if delta >= 0.05: return "meaningful"
    return "noise"

('zeroshot', 'L-1') (142588, 768)
('cpt', 'L-1') (142588, 768)
('zeroshot', 'L0') (142588, 768)
('cpt', 'L0') (142588, 768)


> **Interpretation — four matrices assembled and aligned (5a).** The four embedding matrices every eval below runs against are now loaded and aligned to the same `cell_index` ordering: the two pre-existing `emb_layer=-1` matrices (zero-shot from an earlier notebook, CPT from the aggregated v2 run) and the two new `emb_layer=0` matrices produced in 4b. All four print the identical shape `(142588, 768)`, and the assert immediately after (checking all four shapes are the same single value) passed silently — meaning row *i* refers to the same physical cell across all four matrices, the precondition every downstream probe and k-NN comparison depends on. This cell also defines the verdict-banding functions used everywhere below (`band_probe`/`band_knn`/`band_sil`), which encode the pre-registered noise/meaningful/decisive/regression thresholds fixed before any CPT result existed — so the labels attached to each eval's Δ in 6b/7b are not chosen after seeing the numbers.

## 6 — Eval #1: substate linear probe

### 6a — Held-out substate composition audit (a single-donor substate is a null, not a win)

The probe can only be trusted where the held-out set actually *has* the substate. The contract requires reporting, per lineage and per substate, how many held-out donors and which studies carry it — a substate that is near-absent or single-donor in the test set is unevaluable and is reported as underpowered, never as a win. The `intermediate` bucket is excluded from the binary probe (microglia: activated vs homeostatic; astrocyte: reactive vs resting) and shown here for completeness.

In [9]:
BINARY = {"microglia": ("homeostatic", "activated"), "astrocyte": ("resting", "reactive")}

UNDERPOWERED_1 = {}
print("held-out substate composition (donors / studies driving eval #1):")
for lin, (neg, pos) in BINARY.items():
    m = is_test & (lineage == lin)
    sub = obs.loc[m, ["substate", "donor_id", "study_id"]]
    print(f"\n[{lin}] binary classes = {neg} vs {pos}  (intermediate excluded)")
    underpowered = False
    for s in (neg, pos, "intermediate"):
        ss = sub[sub["substate"] == s]
        nd = ss["donor_id"].nunique()
        studies = ss["study_id"].value_counts().to_dict()
        is_thin = s != "intermediate" and nd < 3
        underpowered = underpowered or is_thin
        flag = "" if s == "intermediate" else ("  [UNDERPOWERED]" if is_thin else "")
        print(f"  {s:12s}: {len(ss):5d} cells | {nd} donors | {studies}{flag}")
    UNDERPOWERED_1[lin] = underpowered

held-out substate composition (donors / studies driving eval #1):

[microglia] binary classes = homeostatic vs activated  (intermediate excluded)
  homeostatic :  5538 cells | 21 donors | {'SEA-AD': 2843, 'Li2025': 2159, 'Haney2024': 536}
  activated   :  1883 cells | 22 donors | {'SEA-AD': 1389, 'Li2025': 250, 'Haney2024': 244}
  intermediate:  2508 cells | 21 donors | {'SEA-AD': 1431, 'Li2025': 802, 'Haney2024': 275}

[astrocyte] binary classes = resting vs reactive  (intermediate excluded)
  resting     :  6046 cells | 22 donors | {'Li2025': 2930, 'SEA-AD': 2578, 'Haney2024': 538}
  reactive    :  4199 cells | 22 donors | {'SEA-AD': 3023, 'Haney2024': 802, 'Li2025': 374}
  intermediate:  3627 cells | 19 donors | {'SEA-AD': 2241, 'Haney2024': 765, 'Li2025': 621}


> **Interpretation — held-out substate composition audit (6a).** Before trusting any probe result, this audits whether the held-out test set actually has enough cells and donors of each substate to make the probe meaningful. For microglia (homeostatic vs. activated) and astrocyte (resting vs. reactive), every one of the four binary classes clears the power floor: 21–22 distinct donors each (the underpowered threshold is <3 donors), and all four classes draw cells from all three studies rather than being carried by a single cohort. The `intermediate` bucket (2,508 micro cells / 21 donors; 3,627 astro cells / 19 donors) is shown for transparency but deliberately excluded from the binary probe in 6b. Because no class tripped the underpowered flag here, the probe results that follow get a real noise/meaningful/decisive/regression verdict rather than being forced to "underpowered" — this eval has a genuine chance to detect a CPT effect if one exists.

### 6b — Probe at both extraction points, Δ vs zero-shot with contract bands

A logistic-regression probe is trained on the train-split cells' embedding and scored (balanced accuracy) on the disjoint held-out donors — donor-held-out by construction, so a prediction cannot be a memorised donor identity. The same probe is fit separately on the zero-shot and CPT embeddings at each layer; the eval verdict is the **CPT − zero-shot** gain against the fixed bands (noise ca. 2% / meaningful ≥5% / decisive ≥10% / regression >2% drop). The L−1-vs-L0 comparison shows whether reading one layer deeper — where layer 11's absorbed gain lives — surfaces structure the standard readout misses.

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score

def probe_bacc(X, y_train_mask, y_test_mask, y):
    sc_ = StandardScaler().fit(X[y_train_mask])
    clf = LogisticRegression(max_iter=2000, class_weight="balanced")
    clf.fit(sc_.transform(X[y_train_mask]), y[y_train_mask])
    pred = clf.predict(sc_.transform(X[y_test_mask]))
    return balanced_accuracy_score(y[y_test_mask], pred)

EVAL1 = {}
for lin, (neg, pos) in BINARY.items():
    in_lin  = lineage == lin
    is_bin  = np.isin(substate, [neg, pos])
    tr_mask = is_train & in_lin & is_bin
    te_mask = is_test  & in_lin & is_bin
    y = (substate == pos).astype(int)   # 1 = activated / reactive
    print(f"\n[{lin}] train {int(tr_mask.sum())} / test {int(te_mask.sum())} cells")
    for layer in LAYERS:
        b_zs  = probe_bacc(EMB[("zeroshot", layer)], tr_mask, te_mask, y)
        b_cpt = probe_bacc(EMB[("cpt", layer)],      tr_mask, te_mask, y)
        delta = (b_cpt - b_zs) * 100
        # a substate near-absent/single-donor in the held-out set is unevaluable -- reported as
        # underpowered, never as a win (contract §eval1 mandatory confound reporting).
        verdict = "underpowered" if UNDERPOWERED_1[lin] else band_probe(delta)
        EVAL1[(lin, layer)] = {"bacc_zeroshot": round(float(b_zs), 4),
                               "bacc_cpt": round(float(b_cpt), 4),
                               "delta_pp": round(float(delta), 2),
                               "verdict": verdict}
        print(f"  {layer}: zero-shot {b_zs:.3f} -> CPT {b_cpt:.3f} | Δ {delta:+.2f} pp | {verdict}")


[microglia] train 26548 / test 7421 cells
  L-1: zero-shot 0.907 -> CPT 0.906 | Δ -0.12 pp | noise
  L0: zero-shot 0.909 -> CPT 0.907 | Δ -0.16 pp | noise

[astrocyte] train 53839 / test 10245 cells
  L-1: zero-shot 0.778 -> CPT 0.777 | Δ -0.17 pp | noise
  L0: zero-shot 0.786 -> CPT 0.787 | Δ +0.12 pp | noise


> **Interpretation — substate probe results (6b).** A logistic-regression probe was trained separately on each of the four embedding matrices (zero-shot/CPT × L−1/L0), fit only on train-split cells and scored with balanced accuracy — so the class imbalance between, e.g., 5,538 homeostatic and 1,883 activated test cells doesn't inflate the number — on the disjoint held-out donors. All four zero-shot baselines are already high (0.778–0.909): even without any CPT fine-tuning, Geneformer's raw embeddings already linearly separate these substates well. The CPT-vs-zero-shot deltas are tiny in both directions (−0.17pp to +0.12pp) and every one lands in the pre-registered "noise" band (threshold ±2pp) — including at L0, the layer specifically added to this notebook's design to catch whatever the head-ablation diagnostic found concentrated in the last encoder layer. So this eval finds no probe-detectable substate benefit from CPT at either extraction point, in either lineage. One caveat: L0 only sees layer-11's share of what CPT changed (roughly 43% of the absorbed loss gain, per the head-ablation result); the head's share (~39%) never becomes an embedding at all, so this result cannot rule out CPT changing head-only structure — it only rules out a probe-detectable change at the two extraction points actually available to read.

## 7 — Eval #2: APOE-carrier recovery (Stanton core)

### 7a — APOE composition and confound audit (study × region of the held-out carriers)

The load-bearing biological axis: does CPT make E4-carrier status more recoverable *within* each lineage? First the mandatory confound audit — a "recovery" that is really study or region leakage is a null. Carrier vs non-carrier is binary; `e2` cells are excluded per the locked label definition. This cell reports, per lineage, the held-out carrier / non-carrier counts broken down by study and region, and flags any lineage where a class is too thin to evaluate.

In [11]:
APOE_KEEP = {"carrier", "noncarrier"}   # e2 excluded per locked E4 definition

UNDERPOWERED_2 = {}
print("held-out APOE composition (study x region driving eval #2):")
for lin in ("microglia", "astrocyte"):
    m = is_test & (lineage == lin) & np.isin(apoe, list(APOE_KEEP))
    sub = obs.loc[m, ["apoe_carrier", "study_id", "region", "donor_id"]]
    print(f"\n[{lin}] held-out carrier/noncarrier cells: {len(sub)}")
    underpowered = False
    for cls in ("carrier", "noncarrier"):
        cc = sub[sub["apoe_carrier"] == cls]
        nd = cc["donor_id"].nunique()
        is_thin = nd < 3
        underpowered = underpowered or is_thin
        flag = "  [UNDERPOWERED]" if is_thin else ""
        print(f"  {cls:11s}: {len(cc):5d} cells | {nd} donors{flag}")
    print("  study x region:")
    print(sub.groupby(["study_id", "region"], observed=True).size().to_string())
    UNDERPOWERED_2[lin] = underpowered

held-out APOE composition (study x region driving eval #2):

[microglia] held-out carrier/noncarrier cells: 9108
  carrier    :  4198 cells | 11 donors
  noncarrier :  4910 cells | 10 donors
  study x region:
study_id   region         
Li2025     temporal cortex    3211
SEA-AD     MTG                4842
Haney2024  unknown            1055

[astrocyte] held-out carrier/noncarrier cells: 13448
  carrier    :  7009 cells | 11 donors
  noncarrier :  6439 cells | 10 donors
  study x region:
study_id   region         
Li2025     temporal cortex    3925
SEA-AD     MTG                7418
Haney2024  unknown            2105


> **Interpretation — APOE composition and confound audit (7a).** The same power-and-confound audit as 6a, now for the APOE-carrier axis. Carrier vs. noncarrier held-out counts clear the power floor comfortably in both lineages (10–11 distinct donors per class, well above the <3-donor underpowered threshold) — notably more than an earlier, now-superseded feasibility estimate had suggested, so this eval is better-powered than expected going in. The study × region breakdown is meant to catch the confound the contract cares about most: if carriers and non-carriers were concentrated in different studies, an apparent "recovery" could just be the classifier learning to tell studies apart rather than genotype. As printed, though, this table pools carrier and noncarrier together (it groups by study × region only, not by `apoe_carrier`) — so it shows which studies dominate the held-out set (SEA-AD contributes roughly half of both lineages’ cells, reflecting its larger size in this cohort) but does not actually show whether any study is carrier-skewed or noncarrier-skewed. That is a real gap in this audit as implemented: the class-resolved breakdown needed to fully rule out study-leakage is not printed here. Given every eval #2 verdict downstream comes back null, no conclusion in this notebook hinges on that gap — there is no "win" it could be quietly explaining away — but it should be closed (add `apoe_carrier` to the groupby) before this cell is trusted as a complete confound check in a future run where a real effect is found. The caveat from 2b still applies regardless: every Haney2024 cell (1,055 microglia / 2,105 astrocyte) carries the placeholder region label "unknown," so the region axis of whatever confound check runs is uninformative for roughly 12% (micro) / 16% (astro) of the held-out carriers/noncarriers.

### 7b — Silhouette + k-NN within astro / within micro, at both extraction points

Two metrics, reported together per the contract. **k-NN** fits on the train-split cells and predicts held-out carrier status (balanced accuracy, donor-disjoint) — this is the **load-bearing** metric here. **Silhouette** of the binary E4 label measures how separated carriers and non-carriers are in the held-out embedding, but at these donor counts it is computed on few cells and is scale- and density-sensitive, so it is treated as **corroborating-only**: it is still scored against its band, but a silhouette move is not read as a verdict on its own — it either agrees with the k-NN read or it is noise. Both are scored as CPT − zero-shot against their own bands — silhouette (noise 0.02 / meaningful ≥0.05 / decisive ≥0.10 / regression Δ<0) and k-NN (noise 3–5% / meaningful ≥5% / decisive ≥10% / regression >3% drop) — at L−1 and L0. Given the thin held-out donor counts (carrier 7 / non-carrier 10 across the whole test set), wide swings are expected and N=3 seeds will not narrow them; the composition audit above governs what is even evaluable.

In [12]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import silhouette_score

def apoe_metrics(X, tr_mask, te_mask, y):
    sc_ = StandardScaler().fit(X[tr_mask])
    Xtr, Xte = sc_.transform(X[tr_mask]), sc_.transform(X[te_mask])
    knn = KNeighborsClassifier(n_neighbors=15).fit(Xtr, y[tr_mask])
    bacc = balanced_accuracy_score(y[te_mask], knn.predict(Xte))
    sil = silhouette_score(Xte, y[te_mask]) if len(np.unique(y[te_mask])) == 2 else float("nan")
    return bacc, sil

EVAL2 = {}
for lin in ("microglia", "astrocyte"):
    in_lin = (lineage == lin) & np.isin(apoe, list(APOE_KEEP))
    tr_mask = is_train & in_lin
    te_mask = is_test  & in_lin
    y = (apoe == "carrier").astype(int)
    print(f"\n[{lin}] train {int(tr_mask.sum())} / test {int(te_mask.sum())} cells")
    for layer in LAYERS:
        k_zs, s_zs   = apoe_metrics(EMB[("zeroshot", layer)], tr_mask, te_mask, y)
        k_cpt, s_cpt = apoe_metrics(EMB[("cpt", layer)],      tr_mask, te_mask, y)
        dk = (k_cpt - k_zs) * 100
        ds = s_cpt - s_zs
        # a class too thin to evaluate is unevaluable -- reported as underpowered, never as a win
        # (contract §eval2 mandatory confound reporting), for both k-NN and silhouette.
        knn_verdict = "underpowered" if UNDERPOWERED_2[lin] else band_knn(dk)
        sil_verdict = "underpowered" if UNDERPOWERED_2[lin] else band_sil(ds)
        EVAL2[(lin, layer)] = {
            "knn_bacc_zeroshot": round(float(k_zs), 4), "knn_bacc_cpt": round(float(k_cpt), 4),
            "knn_delta_pp": round(float(dk), 2), "knn_verdict": knn_verdict,
            "sil_zeroshot": round(float(s_zs), 4), "sil_cpt": round(float(s_cpt), 4),
            "sil_delta": round(float(ds), 4), "sil_verdict": sil_verdict}
        print(f"  {layer}: k-NN {k_zs:.3f}->{k_cpt:.3f} (Δ{dk:+.2f}pp {knn_verdict}) | "
              f"sil {s_zs:+.3f}->{s_cpt:+.3f} (Δ{ds:+.3f} {sil_verdict})")


[microglia] train 30499 / test 9108 cells
  L-1: k-NN 0.458->0.459 (Δ+0.12pp noise) | sil +0.061->+0.062 (Δ+0.000 noise)
  L0: k-NN 0.448->0.447 (Δ-0.15pp noise) | sil +0.075->+0.072 (Δ-0.003 regression)

[astrocyte] train 49620 / test 13448 cells
  L-1: k-NN 0.402->0.402 (Δ-0.01pp noise) | sil +0.048->+0.048 (Δ-0.001 regression)
  L0: k-NN 0.398->0.401 (Δ+0.24pp noise) | sil +0.065->+0.062 (Δ-0.003 regression)


> **Interpretation — the Stanton-core result (7b).** A k-NN classifier (load-bearing) and silhouette score (corroborating-only, since these donor counts make it a noisy, scale-sensitive statistic) were each computed on zero-shot and CPT embeddings, at both extraction points, within each lineage. k-NN deltas are all tiny (−0.15pp to +0.24pp) and all land in "noise" — CPT produced no detectable improvement in how separable carrier-vs-noncarrier status is from a nearest-neighbor read of the embedding, at either L−1 or L0, in either lineage. Worth noting beyond the CPT-vs-zero-shot comparison: the absolute k-NN balanced accuracies themselves are all 0.40–0.46 — at or below the 0.50 chance line for a balanced two-class read, for both zero-shot and CPT, in both lineages, at both layers. That is a stronger statement than "CPT did not help": these embeddings do not appear to carry k-NN-recoverable APOE-carrier signal at all, with or without CPT. Silhouette shows small negative deltas (−0.001 to −0.003) in three of the four combinations, which the fixed banding labels "regression" (its threshold is any decrease at all) — but the baseline silhouette values themselves are already close to zero (+0.048 to +0.075, on a scale where 1.0 is perfect separation and 0 is none), so these are small movements around an already near-null starting point, not a genuine loss of previously-present structure. Because silhouette is explicitly corroborating-only here, the k-NN read is what should be trusted, and it agrees across all four combinations: no APOE-axis recovery from CPT, at either extraction point.

## 8 — Summary and handoff

### 8a — Verdict table, append the audit trace, print the commit commands

The per-eval, per-lineage, per-layer verdicts assembled into one table, then written to `outputs/audit_report.json` under `geneformer_cpt_evals`. The headline read is the **L−1 vs L0** contrast: L−1 is the verdict the pipeline's actual readout supports; the L−1→L0 change is the partial (layer-11-only, head-blind) eval-space view of the absorbed gain. A `SMOKE` run does **not** write the audit trace (its numbers are plumbing, not results). Commit commands are printed for the WSL side, since Colab has no git credentials.

In [13]:
import shlex

print("=== EVAL #1 (substate probe) — balanced-acc gain, CPT vs zero-shot ===")
for (lin, layer), r in EVAL1.items():
    print(f"  {lin:10s} {layer:4s}: {r['bacc_zeroshot']:.3f} -> {r['bacc_cpt']:.3f}  Δ{r['delta_pp']:+.2f}pp  [{r['verdict']}]")
print("\n=== EVAL #2 (APOE recovery) — k-NN gain / silhouette gain ===")
for (lin, layer), r in EVAL2.items():
    print(f"  {lin:10s} {layer:4s}: kNN Δ{r['knn_delta_pp']:+.2f}pp [{r['knn_verdict']}] | "
          f"sil Δ{r['sil_delta']:+.3f} [{r['sil_verdict']}]")

if SMOKE:
    print("\n[SMOKE] audit trace NOT written (plumbing run).")
else:
    AUDIT_REPORT_PATH = os.path.join(REPO_PATH, "outputs", "audit_report.json")
    with open(AUDIT_REPORT_PATH) as f:
        report = json.load(f)
    report["geneformer_cpt_evals"] = {
        "status": "computed", "date": TODAY, "regime": "aggregated", "seed": SEED,
        "reads_run": "geneformer_cpt_aggregated_v2", "extraction_points": {"L-1": "second-to-last", "L0": "last layer"},
        "geneformer_commit": GENEFORMER_COMMIT,
        "eval1_substate_probe": {f"{lin}|{layer}": r for (lin, layer), r in EVAL1.items()},
        "eval2_apoe_recovery":  {f"{lin}|{layer}": r for (lin, layer), r in EVAL2.items()},
        "embedding_files": {
            "zeroshot_L-1": os.path.relpath(ZS_L1_PATH, DRIVE_ROOT),
            "cpt_L-1":      os.path.relpath(CPT_L1_PATH, DRIVE_ROOT),
            "zeroshot_L0":  os.path.relpath(ZS_L0_PATH, DRIVE_ROOT),
            "cpt_L0":       os.path.relpath(CPT_L0_PATH, DRIVE_ROOT)},
        "note": ("L-1 = pipeline readout; L0 captures layer-11's absorbed share only (head unreachable as "
                 "embedding). Eval #2: k-NN is load-bearing; silhouette corroborating-only at these donor counts."),
    }
    with open(AUDIT_REPORT_PATH, "w") as f:
        json.dump(report, f, indent=2)
    print("\naudit trace appended ->", AUDIT_REPORT_PATH)
    rel = [os.path.relpath(p, REPO_PATH) for p in (FREEZE_PATH, ENV_JSON_PATH, AUDIT_REPORT_PATH)]
    print("\n=== Commit + push (from WSL) ===")
    print("  git add " + " ".join(shlex.quote(r) for r in rel))
    print("  git commit -m 'colab_12: Geneformer CPT evals #1 (substate probe) + #2 (APOE) at L-1 and L0'")
    print("  git push")

=== EVAL #1 (substate probe) — balanced-acc gain, CPT vs zero-shot ===
  microglia  L-1 : 0.907 -> 0.906  Δ-0.12pp  [noise]
  microglia  L0  : 0.909 -> 0.907  Δ-0.16pp  [noise]
  astrocyte  L-1 : 0.778 -> 0.777  Δ-0.17pp  [noise]
  astrocyte  L0  : 0.786 -> 0.787  Δ+0.12pp  [noise]

=== EVAL #2 (APOE recovery) — k-NN gain / silhouette gain ===
  microglia  L-1 : kNN Δ+0.12pp [noise] | sil Δ+0.000 [noise]
  microglia  L0  : kNN Δ-0.15pp [noise] | sil Δ-0.003 [regression]
  astrocyte  L-1 : kNN Δ-0.01pp [noise] | sil Δ-0.001 [regression]
  astrocyte  L0  : kNN Δ+0.24pp [noise] | sil Δ-0.003 [regression]

audit trace appended -> /content/ad-glia-fm-prep/outputs/audit_report.json

=== Commit + push (from WSL) ===
  git add outputs/software_versions/colab_12_2026-07-20_pip_freeze.txt outputs/software_versions/colab_12_2026-07-20_env.json outputs/audit_report.json
  git commit -m 'colab_12: Geneformer CPT evals #1 (substate probe) + #2 (APOE) at L-1 and L0'
  git push


> **Interpretation — verdict table, audit write, headline result (8a).** The verdict table reproduces every number from 6b and 7b exactly, confirming nothing drifted between computing the individual evals and assembling the summary. The headline: both eval #1 (substate probe) and eval #2 (APOE recovery, the Stanton-core axis) are null at both extraction points, in both lineages — CPT’s loss improvement, established separately as real and concentrated in the head and last encoder layer, is not showing up as anything a linear probe or k-NN can decode through either the standard L−1 readout or the deeper L0 view. Eval #2’s null is actually a floor null, not just a flat one: the k-NN balanced accuracies sit at or below chance (0.40–0.46) regardless of CPT (see 7b) — APOE-carrier status does not appear to be k-NN-recoverable from these Geneformer embeddings at all, CPT or zero-shot. The audit trace was appended to `outputs/audit_report.json` under `geneformer_cpt_evals`, carrying every verdict plus a note documenting L0’s partial-view limitation and silhouette’s corroborating-only status, so this result is traceable independent of the notebook itself. This is still an N=1 pilot (one aggregated-regime CPT run) and does not establish whether the null generalizes across seeds or regimes; eval #3 / detector #2 (a separate notebook, not yet run) still need to check whether CPT caused any catastrophic forgetting on top of this null before the aggregated-regime result can be considered complete.